# OpenPlaque — directional outer-wall / fat-interface PCAT

This notebook keeps the accepted RCA centerline and 10–50 mm interval fixed, but replaces the circular outer-wall approximation with a **direction-dependent vessel–fat interface** estimated independently along radial rays.

Research prototype only; this is not Caristo FAI-Score.


In [ ]:
# FIRST EXECUTABLE CELL
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!rm -rf /content/OpenPlaque
!git clone -q --branch pcat-radial-boundary-diagnostic-from-main --single-branch https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
%pip -q install pydicom SimpleITK scipy matplotlib pandas


In [ ]:
import sys, shutil, math, zipfile
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, SimpleITK as sitk
from scipy import ndimage as ndi
from scipy.spatial import cKDTree
sys.path.insert(0,'/content/OpenPlaque/src')
from openplaque.study import OpenPlaqueStudy
ROOT=Path('/content/drive/MyDrive/OpenPlaque')
BASE=ROOT/'PCAT_RCA_10_50'
OUT=ROOT/'PCAT_RCA_10_50_Directional_OuterWall'; OUT.mkdir(parents=True,exist_ok=True)
FAT_LO,FAT_HI=-190.,-30.
SEG0,SEG1=10.,50.
N_RAYS=48
RSTEP=0.20
RMAX=7.0
FAT_RUN_SAMPLES=3
MAX_WALL_PLUS_PLAQUE_MM=3.0
print('Output:',OUT)


## 1. Load source CCTA and fixed centerline/radius model


In [ ]:
dz=ROOT/'Full_DICOM.zip'; lz=Path('/content/Full_DICOM.zip')
if not lz.exists() or lz.stat().st_size!=dz.stat().st_size: shutil.copyfile(dz,lz)
shutil.rmtree('/content/full_dicom_pcat_directional',ignore_errors=True)
study=OpenPlaqueStudy(str(lz),extract_root='/content/full_dicom_pcat_directional')
img,ct,_=study.load_series(7); ct=np.asarray(ct)
sp_xyz=np.array(img.GetSpacing(),float); sp_zyx=sp_xyz[::-1]; voxel_mm3=float(np.prod(sp_xyz))
cp=BASE/'rca_centerline_smoothed_zyx.csv'; rp=BASE/'pcat_local_radius_profile.csv'
if not cp.exists() or not rp.exists(): raise FileNotFoundError('Run RCA 10–50 mm PCAT prototype first.')
cl=pd.read_csv(cp); rad=pd.read_csv(rp)
arc=cl.arc_mm.to_numpy(float); pts=cl[['z','y','x']].to_numpy(float); pts_mm=pts*sp_zyx
lum_all=np.interp(arc,rad.arc_mm.to_numpy(float),rad.lumen_radius_mm.to_numpy(float))
m=(arc>=SEG0)&(arc<=SEG1)
seg_arc=arc[m]; seg_zyx=pts[m]; seg_mm=pts_mm[m]; seg_lum=lum_all[m]
print('Segment points:',len(seg_arc),'arc',seg_arc.min(),'to',seg_arc.max())
print('Mean lumen radius mm:',seg_lum.mean())


## 2. Estimate directional vessel–fat interfaces

For each centerline point and ray, the interface is the first sustained adipose run (≥0.6 mm at the default sampling) encountered outside the fixed lumen radius, provided it begins within 3 mm of that lumen radius. Directions with no nearby fat are deliberately rejected.


In [ ]:
def tangent(i):
    a=max(0,i-2); b=min(len(seg_mm)-1,i+2)
    t=seg_mm[b]-seg_mm[a]; return t/max(np.linalg.norm(t),1e-9)
def basis(t):
    ref=np.array([1.,0.,0.]);
    if abs(np.dot(t,ref))>.85: ref=np.array([0.,1.,0.])
    u=np.cross(t,ref); u/=max(np.linalg.norm(u),1e-9)
    v=np.cross(t,u); v/=max(np.linalg.norm(v),1e-9)
    return u,v
T=np.vstack([tangent(i) for i in range(len(seg_mm))])
U=[]; V=[]
for t in T:
    u,v=basis(t); U.append(u); V.append(v)
U=np.vstack(U); V=np.vstack(V)
def sample_mm(qmm):
    q=np.asarray(qmm,float)/sp_zyx
    return ndi.map_coordinates(ct.astype(np.float32,copy=False),[q[:,0],q[:,1],q[:,2]],order=1,mode='nearest')
rvals=np.arange(0,RMAX+1e-9,RSTEP)
angles=np.linspace(0,2*np.pi,N_RAYS,endpoint=False)
iface=np.full((len(seg_mm),N_RAYS),np.nan,float)
for i,p in enumerate(seg_mm):
    for k,a in enumerate(angles):
        d=np.cos(a)*U[i]+np.sin(a)*V[i]
        q=p[None,:]+rvals[:,None]*d[None,:]
        hu=sample_mm(q)
        fat=(hu>=FAT_LO)&(hu<=FAT_HI)
        j0=int(np.searchsorted(rvals,seg_lum[i]+0.2))
        found=None
        for j in range(j0,len(rvals)-FAT_RUN_SAMPLES+1):
            if np.all(fat[j:j+FAT_RUN_SAMPLES]): found=j; break
        if found is not None:
            rr=float(rvals[found])
            if rr-seg_lum[i] <= MAX_WALL_PLUS_PLAQUE_MM: iface[i,k]=rr
accepted=np.isfinite(iface)
acc_frac=accepted.mean(axis=1)
median_iface=np.nanmedian(iface,axis=1)
valid_point=np.sum(accepted,axis=1)>=8
local_diam=np.where(valid_point,np.clip(2*median_iface,3.0,8.0),np.nan)
iface_df=[]
for i,s in enumerate(seg_arc):
    iface_df.append({'arc_mm':s,'lumen_radius_mm':seg_lum[i],'accepted_ray_fraction':acc_frac[i],
                     'median_interface_radius_mm':median_iface[i],
                     'median_wall_plus_plaque_mm':median_iface[i]-seg_lum[i] if np.isfinite(median_iface[i]) else np.nan,
                     'local_sampling_diameter_mm':local_diam[i]})
iface_summary=pd.DataFrame(iface_df)
iface_summary.to_csv(OUT/'directional_interface_by_arc.csv',index=False)
display(iface_summary.head())
print('Median accepted-ray fraction:',float(np.nanmedian(acc_frac)))
print('Valid centerline fraction:',float(valid_point.mean()))


## 3. Build an irregular 3-D perivascular shell from the directional interfaces


In [ ]:
pad=12.0
lo=np.floor(np.min(seg_zyx,axis=0)-pad/sp_zyx).astype(int); hi=np.ceil(np.max(seg_zyx,axis=0)+pad/sp_zyx).astype(int)+1
lo=np.maximum(lo,0); hi=np.minimum(hi,np.array(ct.shape))
crop=ct[lo[0]:hi[0],lo[1]:hi[1],lo[2]:hi[2]]
zz,yy,xx=np.indices(crop.shape)
g=np.stack([zz+lo[0],yy+lo[1],xx+lo[2]],axis=-1).reshape(-1,3).astype(float)
gmm=g*sp_zyx
tree=cKDTree(seg_mm); _,ni=tree.query(gmm,k=1,workers=-1); ni=ni.astype(int)
vec=gmm-seg_mm[ni]
tt=T[ni]; uu=U[ni]; vv=V[ni]
axial=np.sum(vec*tt,axis=1)
perp=vec-axial[:,None]*tt
rr=np.linalg.norm(perp,axis=1)
aa=np.mod(np.arctan2(np.sum(perp*vv,axis=1),np.sum(perp*uu,axis=1)),2*np.pi)
sector=np.mod(np.rint(aa/(2*np.pi)*N_RAYS).astype(int),N_RAYS)
ir=iface[ni,sector]; thick=local_diam[ni]
ok=np.isfinite(ir)&np.isfinite(thick)&(np.abs(axial)<=0.8)
shell=ok&(rr>ir)&(rr<=ir+thick)
hu=crop.reshape(-1).astype(float)
fat=shell&(hu>=FAT_LO)&(hu<=FAT_HI)
outward=rr-ir
shell3=shell.reshape(crop.shape); fat3=fat.reshape(crop.shape)
print('Shell voxels:',int(shell.sum()),'fat voxels:',int(fat.sum()))


## 4. Quantify directional PCAT and compare with the circular-wall prototype


In [ ]:
vals=hu[fat]
rad_rows=[]
for b in np.arange(0,6,0.5):
    sm=shell&(outward>=b)&(outward<b+0.5)
    fm=sm&(hu>=FAT_LO)&(hu<=FAT_HI); hm=sm&(hu>FAT_HI); lm=sm&(hu<FAT_LO)
    fv=hu[fm]
    rad_rows.append({'radial_start_mm':b,'radial_end_mm':b+0.5,'shell_voxels':int(sm.sum()),
                     'fat_voxels':int(fm.sum()),'high_gt_minus30_voxels':int(hm.sum()),'low_lt_minus190_voxels':int(lm.sum()),
                     'fat_fraction':float(fm.sum()/max(1,sm.sum())),
                     'mean_hu':float(np.mean(fv)) if len(fv) else np.nan})
radial=pd.DataFrame(rad_rows); radial.to_csv(OUT/'directional_pcat_radial_0p5mm.csv',index=False)
long_rows=[]
nearest_arc=seg_arc[ni]
for b in range(10,50):
    fm=fat&(nearest_arc>=b)&(nearest_arc<b+1); fv=hu[fm]
    long_rows.append({'arc_start_mm':b,'arc_end_mm':b+1,'fat_voxels':int(fm.sum()),'mean_hu':float(np.mean(fv)) if len(fv) else np.nan})
longitudinal=pd.DataFrame(long_rows); longitudinal.to_csv(OUT/'directional_pcat_longitudinal.csv',index=False)
first6=radial.iloc[:6]; good=first6.fat_voxels.to_numpy()>=100
slope=float(np.polyfit(((first6.radial_start_mm+first6.radial_end_mm)/2).to_numpy()[good],first6.mean_hu.to_numpy()[good],1)[0]) if good.sum()>=3 else np.nan
near=radial[(radial.radial_start_mm>=0)&(radial.radial_end_mm<=2)]
outer=radial[(radial.radial_start_mm>=2)&(radial.radial_end_mm<=4)]
def weighted_mean(df):
    w=df.fat_voxels.to_numpy(float); x=df.mean_hu.to_numpy(float); m=np.isfinite(x)&(w>0); return float(np.sum(x[m]*w[m])/np.sum(w[m])) if np.any(m) else np.nan
near_mean=weighted_mean(near); outer_mean=weighted_mean(outer)
circ_mean=np.nan
ps=BASE/'pcat_summary.csv'
if ps.exists():
    z=pd.read_csv(ps);
    for c in ['pcat_mean_hu','mean_pcat_hu','openplaque_pcat_mean_hu']:
        if c in z.columns: circ_mean=float(z.iloc[0][c]); break
summary=pd.DataFrame([{'directional_pcat_mean_hu':float(np.mean(vals)),'directional_pcat_median_hu':float(np.median(vals)),
                       'directional_pcat_sd_hu':float(np.std(vals)),'fat_voxels':int(fat.sum()),
                       'fat_volume_ml':float(fat.sum()*voxel_mm3/1000),'shell_volume_ml':float(shell.sum()*voxel_mm3/1000),
                       'median_accepted_ray_fraction':float(np.nanmedian(acc_frac)),'valid_centerline_fraction':float(valid_point.mean()),
                       'median_wall_plus_plaque_mm':float(np.nanmedian(median_iface-seg_lum)),
                       'near_0_2mm_mean_hu':near_mean,'outer_2_4mm_mean_hu':outer_mean,
                       'outer_minus_near_hu':outer_mean-near_mean,'radial_slope_0_3_hu_per_mm':slope,
                       'circular_reference_mean_hu':circ_mean}])
summary.to_csv(OUT/'directional_pcat_summary.csv',index=False)
display(summary.T); display(radial)


## 5. Cross-sectional QC and comparison figures


In [ ]:
targets=[10,20,30,40,50]
fig,axs=plt.subplots(1,5,figsize=(20,4))
grid=np.linspace(-7,7,141)
A,B=np.meshgrid(grid,grid,indexing='xy')
for ax,targ in zip(axs,targets):
    i=int(np.argmin(abs(seg_arc-targ))); p=seg_mm[i]
    q=p[None,None,:]+A[:,:,None]*U[i][None,None,:]+B[:,:,None]*V[i][None,None,:]
    imv=sample_mm(q.reshape(-1,3)).reshape(A.shape)
    ax.imshow(imv,cmap='gray',origin='lower',extent=[-7,7,-7,7],vmin=-200,vmax=700)
    goodk=np.where(np.isfinite(iface[i]))[0]
    if len(goodk):
        x=iface[i,goodk]*np.cos(angles[goodk]); y=iface[i,goodk]*np.sin(angles[goodk])
        ax.scatter(x,y,s=9,label='directional interface')
    circ=seg_lum[i]+0.75; th=np.linspace(0,2*np.pi,200)
    ax.plot(circ*np.cos(th),circ*np.sin(th),'--',linewidth=1,label='old circular wall')
    ax.scatter([0],[0],s=12)
    ax.set_title(f'{targ} mm  accepted={acc_frac[i]:.0%}')
    ax.set_xlim(-7,7); ax.set_ylim(-7,7); ax.axis('off')
axs[0].legend(fontsize=7,loc='lower left')
plt.tight_layout(); p1=OUT/'01_directional_interface_qc.png'; fig.savefig(p1,dpi=180,bbox_inches='tight'); plt.show(); plt.close(fig)
fig,axs=plt.subplots(1,3,figsize=(15,4))
axs[0].plot(seg_arc,acc_frac); axs[0].set_xlabel('arc mm'); axs[0].set_ylabel('accepted ray fraction'); axs[0].set_title('Directional interface coverage')
axs[1].plot(seg_arc,median_iface-seg_lum); axs[1].set_xlabel('arc mm'); axs[1].set_ylabel('mm beyond lumen radius'); axs[1].set_title('Median wall/plaque-to-fat distance')
axs[2].plot((radial.radial_start_mm+radial.radial_end_mm)/2,radial.mean_hu,marker='o',label='directional')
pr=BASE/'pcat_radial_profile.csv'
if pr.exists():
    cr=pd.read_csv(pr)
    xcol=next((c for c in cr.columns if 'radial' in c and 'start' in c),None); ycol=next((c for c in cr.columns if 'mean' in c and 'hu' in c),None)
    if xcol and ycol: axs[2].plot(cr[xcol].to_numpy(float)+0.5,cr[ycol],marker='o',label='circular')
axs[2].set_xlabel('mm outward'); axs[2].set_ylabel('PCAT mean HU'); axs[2].set_title('Radial profile'); axs[2].legend()
plt.tight_layout(); p2=OUT/'02_directional_summary.png'; fig.savefig(p2,dpi=180,bbox_inches='tight'); plt.show(); plt.close(fig)
fig,ax=plt.subplots(figsize=(8,5))
x=(radial.radial_start_mm+radial.radial_end_mm)/2
ax.plot(x,radial.fat_fraction,marker='o',label='fat fraction')
high=radial.high_gt_minus30_voxels/np.maximum(radial.shell_voxels,1); low=radial.low_lt_minus190_voxels/np.maximum(radial.shell_voxels,1)
ax.plot(x,high,marker='o',label='> -30 HU fraction'); ax.plot(x,low,marker='o',label='< -190 HU fraction')
ax.set_xlabel('mm outward from directional interface'); ax.set_ylabel('fraction'); ax.set_title('Directional-shell tissue composition'); ax.legend()
plt.tight_layout(); p3=OUT/'03_directional_composition.png'; fig.savefig(p3,dpi=180,bbox_inches='tight'); plt.show(); plt.close(fig)


## 6. Save masks and create one report-back ZIP


In [ ]:
def save_crop_mask(mask,path):
    full=np.zeros(ct.shape,np.uint8)
    full[lo[0]:hi[0],lo[1]:hi[1],lo[2]:hi[2]]=mask.astype(np.uint8)
    out=sitk.GetImageFromArray(full); out.CopyInformation(img); sitk.WriteImage(out,str(path)); del full,out
save_crop_mask(shell3,OUT/'directional_pcat_shell_mask.nii.gz')
save_crop_mask(fat3,OUT/'directional_pcat_fat_mask.nii.gz')
files=['01_directional_interface_qc.png','02_directional_summary.png','03_directional_composition.png',
       'directional_interface_by_arc.csv','directional_pcat_summary.csv','directional_pcat_radial_0p5mm.csv','directional_pcat_longitudinal.csv']
zp=OUT/'PCAT_DIRECTIONAL_OUTERWALL_REPORT_BACK.zip'
with zipfile.ZipFile(zp,'w',zipfile.ZIP_DEFLATED) as z:
    for f in files:
        p=OUT/f
        if p.exists(): z.write(p,arcname=p.name)
print('Report ZIP:',zp)
print('Upload this one file back to ChatGPT:')
print('https://drive.google.com/drive/u/0/search?q=PCAT_DIRECTIONAL_OUTERWALL_REPORT_BACK.zip')
for f in files:
    print('https://drive.google.com/drive/u/0/search?q='+f)
